In [1]:
import os
import cv2
import random
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/AUL"
target_path = r"/mnt/lustre/lalit47/down/class/AUL"

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)

class_names = ["Benign", "Malignant", "Normal"]


def process_and_save(src, dst):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)
    return True


def collect_classification_images():
    data_by_class = defaultdict(list)

    for class_name in class_names:
        image_dir = os.path.join(
            source_path,
            class_name,
            class_name,
            "image"
        )

        if not os.path.isdir(image_dir):
            print(f"Image folder not found: {image_dir}")
            continue

        for file_name in os.listdir(image_dir):

            if not file_name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
                continue

            image_path = os.path.join(image_dir, file_name)
            base_name = os.path.splitext(file_name)[0]

            data_by_class[class_name].append(
                (image_path, base_name, class_name)
            )

    return data_by_class


def stratified_split(data_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_name, samples in data_by_class.items():
        random.shuffle(samples)

        n = len(samples)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_samples = samples[:n_train]
        val_samples = samples[n_train:n_train + n_val]
        test_samples = samples[n_train + n_val:]

        train_data.extend(train_samples)
        val_data.extend(val_samples)
        test_data.extend(test_samples)

        print(
            f"{class_name}: total={n}, "
            f"train={len(train_samples)}, "
            f"val={len(val_samples)}, "
            f"test={len(test_samples)}"
        )

    return train_data, val_data, test_data


def save_classification_split(data, split_name):
    for image_path, base_name, class_name in data:
        class_out_dir = os.path.join(target_path, split_name, class_name)
        os.makedirs(class_out_dir, exist_ok=True)

        image_dst = os.path.join(class_out_dir, base_name + ".png")

        process_and_save(image_path, image_dst)


data_by_class = collect_classification_images()

train_data, val_data, test_data = stratified_split(data_by_class)

save_classification_split(train_data, "train")
save_classification_split(val_data, "val")
save_classification_split(test_data, "test")

print("\nClassification dataset created successfully!")

Benign: total=200, train=140, val=20, test=40
Malignant: total=435, train=304, val=43, test=88
Normal: total=100, train=70, val=10, test=20

Classification dataset created successfully!
